# 🍅 PlantDoc Tomato Disease Classifier — Training Notebook

**Run this on Kaggle (GPU T4 x2) or Google Colab (T4 GPU)**

### What this notebook does:
1. Loads the PlantDoc dataset from Kaggle
2. Filters and validates tomato-only classes
3. Builds augmented train/val datasets
4. Trains MobileNetV2 with transfer learning (freeze → fine-tune)
5. Evaluates per-class precision / recall / F1 + confusion matrix
6. Exports versioned model + config for the inference API

## 0. Environment Setup

In [1]:
import os, sys, json, shutil, datetime
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"Devices     : {tf.config.list_physical_devices()}")

2026-03-08 22:44:14.793997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773009855.003638      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773009855.064356      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773009855.577984      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773009855.578036      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773009855.578040      17 computation_placer.cc:177] computation placer alr

TensorFlow  : 2.19.0
GPU available: False
Devices     : [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-03-08 22:44:46.043609: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 1. Configuration — Edit Here Before Running

In [2]:
# ─── Paths ────────────────────────────────────────────────────────────────────
# Kaggle: dataset is mounted at /kaggle/input/plantdoc-dataset/
# Colab:  adjust if you uploaded manually or mounted Drive
PLANTDOC_ROOT = Path("/kaggle/input/plantdoc-dataset")

# Working directories (writable on Kaggle/Colab)
WORK_DIR      = Path("/kaggle/working")          # change to /content if Colab
DATA_DIR      = WORK_DIR / "data" / "tomato"
MODELS_DIR    = WORK_DIR / "saved_models"
LOGS_DIR      = WORK_DIR / "logs"

for d in [DATA_DIR, MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Model Hyperparameters ────────────────────────────────────────────────────
IMG_SIZE      = (224, 224)   # MobileNetV2 native input size
BATCH_SIZE    = 32
EPOCHS_HEAD   = 15           # Phase 1: train classification head only
EPOCHS_FINETUNE = 10         # Phase 2: unfreeze top layers and fine-tune
LEARNING_RATE = 1e-3
FINETUNE_LR   = 1e-5
DROPOUT       = 0.4
VAL_SPLIT     = 0.2
SEED          = 42

# ─── Uncertainty Thresholds (kept here so API config is generated from same source)
THRESHOLD_CONFIDENT = 0.75   # >= this  → CONFIDENT
THRESHOLD_UNCERTAIN = 0.50   # >= this  → UNCERTAIN  (else FAILED)

# ─── Tomato class mapping ─────────────────────────────────────────────────────
# Keys = exact folder names inside PlantDoc train/test splits
# Values = canonical names used by the API
TOMATO_CLASS_MAP = {
    "Tomato Healthy"            : "Tomato_Healthy",
    "Tomato Early blight leaf"  : "Tomato_Early_Blight",
    "Tomato Late blight leaf"   : "Tomato_Late_Blight",
    "Tomato leaf"               : "Tomato_Healthy",           # alias — PlantDoc has duplicates
    "Tomato leaf bacterial spot": "Tomato_Bacterial_Spot",
    "Tomato leaf mosaic virus"  : "Tomato_Mosaic_Virus",
    "Tomato leaf yellow virus"  : "Tomato_Yellow_Leaf_Curl",
    "Tomato mold leaf"          : "Tomato_Leaf_Mold",
    "Tomato Septoria leaf spot" : "Tomato_Septoria_Leaf_Spot",
    "Tomato two spotted spider mite leaf": "Tomato_Spider_Mite",
}

print("Config loaded ✓")
print(f"Classes targeted: {len(set(TOMATO_CLASS_MAP.values()))}")

Config loaded ✓
Classes targeted: 9


## 2. Dataset Exploration — Understand PlantDoc Structure

In [3]:
def explore_dataset(root: Path):
    """Print PlantDoc folder structure and image counts."""
    print(f"Dataset root: {root}")
    print(f"Top-level folders: {[p.name for p in sorted(root.iterdir()) if p.is_dir()]}\n")

    all_classes = {}
    for split in ["train", "test"]:
        split_dir = root / split
        if not split_dir.exists():
            print(f"  ⚠️  '{split}' folder not found — trying flat structure")
            split_dir = root
        for cls_dir in sorted(split_dir.iterdir()):
            if cls_dir.is_dir():
                n = len(list(cls_dir.glob("*.*")))
                all_classes.setdefault(cls_dir.name, {})[split] = n

    print(f"{'Class':<50} {'Train':>8} {'Test':>8}")
    print("-" * 68)
    tomato_total = 0
    for cls, counts in sorted(all_classes.items()):
        is_tomato = "🍅" if any(cls.lower().startswith("tomato") for _ in [1]) else "  "
        # simpler check:
        is_tomato = "🍅" if cls.lower().startswith("tomato") else "  "
        tr = counts.get("train", 0)
        te = counts.get("test", 0)
        if is_tomato == "🍅":
            tomato_total += tr + te
        print(f"{is_tomato} {cls:<48} {tr:>8} {te:>8}")
    print(f"\nTotal tomato images (approx): {tomato_total}")

explore_dataset(PLANTDOC_ROOT)

Dataset root: /kaggle/input/plantdoc-dataset


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/plantdoc-dataset'

## 3. Filter & Organise Tomato Classes

In [ ]:
def build_tomato_dataset(src_root: Path, dst_root: Path, class_map: dict) -> dict:
    """
    Copies tomato images from PlantDoc into a clean folder structure:
        dst_root/<canonical_class>/<image_file>

    Merges aliased classes (e.g. 'Tomato leaf' → 'Tomato_Healthy').
    Returns a dict {canonical_class: image_count}.
    """
    if dst_root.exists():
        shutil.rmtree(dst_root)  # Clean slate
    dst_root.mkdir(parents=True)

    stats = {}
    copied = skipped = 0

    for split in ["train", "test"]:
        split_dir = src_root / split
        if not split_dir.exists():
            split_dir = src_root  # flat structure fallback

        for src_name, canonical in class_map.items():
            src_cls_dir = split_dir / src_name
            if not src_cls_dir.exists():
                continue

            dst_cls_dir = dst_root / canonical
            dst_cls_dir.mkdir(exist_ok=True)

            for img_path in src_cls_dir.glob("*"):
                if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp"]:
                    skipped += 1
                    continue
                # Unique name: split__srcclass__filename to avoid collisions when merging
                unique_name = f"{split}__{src_name.replace(' ', '_')}__{img_path.name}"
                shutil.copy2(img_path, dst_cls_dir / unique_name)
                copied += 1
                stats[canonical] = stats.get(canonical, 0) + 1

    print(f"✅ Copied {copied} images  |  Skipped {skipped} non-image files")
    print(f"\nClass distribution:")
    total = sum(stats.values())
    for cls, cnt in sorted(stats.items(), key=lambda x: -x[1]):
        bar = "█" * (cnt // 10)
        print(f"  {cls:<40} {cnt:>5}  {bar}")
    print(f"  {'TOTAL':<40} {total:>5}")

    if any(v < 30 for v in stats.values()):
        print("\n⚠️  WARNING: Some classes have < 30 images. Consider merging or excluding them.")

    return stats

class_stats = build_tomato_dataset(PLANTDOC_ROOT, DATA_DIR, TOMATO_CLASS_MAP)
CLASS_NAMES = sorted(class_stats.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f"\nFinal class list ({NUM_CLASSES} classes): {CLASS_NAMES}")

## 4. Data Pipeline — Augmentation & tf.data

In [ ]:
# ─── Augmentation layer (applied only during training) ────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1),
], name="augmentation")

# ─── Load datasets ────────────────────────────────────────────────────────────
train_ds_raw = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,   # Ensure consistent ordering
)

val_ds_raw = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
)

# Confirm class ordering is what we expect
assert train_ds_raw.class_names == CLASS_NAMES, "Class name mismatch!"
print(f"Train batches : {len(train_ds_raw)}")
print(f"Val batches   : {len(val_ds_raw)}")
print(f"Classes       : {CLASS_NAMES}")

In [ ]:
# ─── Preprocessing + performance optimisation ─────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

# MobileNetV2 expects pixels in [-1, 1]
preprocess = keras.applications.mobilenet_v2.preprocess_input

def prepare_train(ds):
    return (
        ds
        .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
        .map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE)
        .cache()
        .shuffle(buffer_size=1000)
        .prefetch(AUTOTUNE)
    )

def prepare_val(ds):
    return (
        ds
        .map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE)
        .cache()
        .prefetch(AUTOTUNE)
    )

train_ds = prepare_train(train_ds_raw)
val_ds   = prepare_val(val_ds_raw)
print("Datasets prepared ✓")

In [ ]:
# ─── Quick visual sanity check ────────────────────────────────────────────────
plt.figure(figsize=(14, 6))
for images, labels in train_ds_raw.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(2, 6, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[labels[i]], fontsize=8)
        plt.axis("off")
plt.suptitle("Sample Training Images (raw, before preprocessing)", y=1.02)
plt.tight_layout()
plt.savefig(LOGS_DIR / "sample_images.png", bbox_inches="tight")
plt.show()

## 5. Model — MobileNetV2 Transfer Learning

We train in **two phases**:
- **Phase 1 (head only):** Base model frozen, only train the new classification head → fast convergence
- **Phase 2 (fine-tune):** Unfreeze top layers of MobileNetV2, train with a very low LR → squeeze out accuracy

In [ ]:
def build_model(num_classes: int, dropout: float = 0.4) -> keras.Model:
    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False   # Frozen for Phase 1

    inputs  = keras.Input(shape=(*IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(dropout)(x)
    x       = layers.Dense(256, activation="relu")(x)
    x       = layers.Dropout(dropout / 2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="tomato_mobilenetv2")
    return model, base_model

model, base_model = build_model(NUM_CLASSES, DROPOUT)

model.compile(
    optimizer=keras.optimizers.Adam(LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary(show_trainable=True)

## 6. Training — Phase 1: Head Only

In [ ]:
callbacks_phase1 = [
    EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        filepath=str(MODELS_DIR / "best_phase1.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

print("=" * 60)
print("PHASE 1 — Training classification head (base frozen)")
print("=" * 60)

history_phase1 = model.fit(
    train_ds,
    epochs=EPOCHS_HEAD,
    validation_data=val_ds,
    callbacks=callbacks_phase1,
)

## 7. Training — Phase 2: Fine-Tune Top Layers

In [ ]:
# Unfreeze the last 30 layers of MobileNetV2
base_model.trainable = True
FINE_TUNE_FROM = len(base_model.layers) - 30

for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after unfreeze: {trainable_count} / {len(model.layers)}")

model.compile(
    optimizer=keras.optimizers.Adam(FINETUNE_LR),   # Much lower LR
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_phase2 = [
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(
        filepath=str(MODELS_DIR / "best_phase2.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

print("=" * 60)
print("PHASE 2 — Fine-tuning top MobileNetV2 layers")
print("=" * 60)

initial_epoch = len(history_phase1.history["loss"])

history_phase2 = model.fit(
    train_ds,
    epochs=initial_epoch + EPOCHS_FINETUNE,
    initial_epoch=initial_epoch,
    validation_data=val_ds,
    callbacks=callbacks_phase2,
)

## 8. Training Curves

In [ ]:
def plot_training_history(h1, h2):
    acc  = h1.history["accuracy"]      + h2.history["accuracy"]
    val  = h1.history["val_accuracy"]  + h2.history["val_accuracy"]
    loss = h1.history["loss"]          + h2.history["loss"]
    vloss= h1.history["val_loss"]      + h2.history["val_loss"]
    phase_boundary = len(h1.history["accuracy"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for ax, train_vals, val_vals, title, ylabel in [
        (ax1, acc, val, "Accuracy", "Accuracy"),
        (ax2, loss, vloss, "Loss", "Loss"),
    ]:
        ax.plot(train_vals, label="Train", linewidth=2)
        ax.plot(val_vals,   label="Val",   linewidth=2)
        ax.axvline(x=phase_boundary, color="red", linestyle="--", alpha=0.7, label="Fine-tune start")
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle("Training History — Phase 1 + Phase 2", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(LOGS_DIR / "training_curves.png", bbox_inches="tight")
    plt.show()
    
    best_val_acc = max(val)
    print(f"\n🏆 Best val accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")
    return best_val_acc

best_val_acc = plot_training_history(history_phase1, history_phase2)

## 9. Evaluation — Per-Class Metrics + Confusion Matrix

In [ ]:
# ─── Collect all predictions on validation set ────────────────────────────────
y_true_all = []
y_pred_all = []
y_prob_all = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)  # shape: (batch, num_classes)
    y_prob_all.extend(preds)
    y_pred_all.extend(np.argmax(preds, axis=1))
    y_true_all.extend(labels.numpy())

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)
y_prob = np.array(y_prob_all)

print(f"Evaluated {len(y_true)} validation images")

In [ ]:
# ─── Classification Report (per-class P / R / F1) ─────────────────────────────
report_str = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3)
print("Per-Class Classification Report")
print("=" * 70)
print(report_str)

# Save as dict for JSON export
report_dict = classification_report(
    y_true, y_pred, target_names=CLASS_NAMES, output_dict=True
)

# Flag classes with F1 < 0.65
print("⚠️  Classes with F1 < 0.65:")
for cls, metrics in report_dict.items():
    if isinstance(metrics, dict) and metrics.get("f1-score", 1.0) < 0.65:
        print(f"   {cls:<40} F1={metrics['f1-score']:.3f}  support={int(metrics['support'])}")

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]  # Row-normalised

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    for ax, data, title, fmt in [
        (axes[0], cm,      "Confusion Matrix (counts)",     "d"),
        (axes[1], cm_norm, "Confusion Matrix (normalised)", ".2f"),
    ]:
        sns.heatmap(
            data,
            annot=True, fmt=fmt, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, linewidths=0.5,
        )
        ax.set_title(title, fontsize=12)
        ax.set_ylabel("True Label")
        ax.set_xlabel("Predicted Label")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.tick_params(axis="y", rotation=0,  labelsize=8)

    plt.suptitle("Tomato Disease — Model Evaluation", fontsize=14)
    plt.tight_layout()
    plt.savefig(LOGS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    return cm

cm = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES)

## 10. Model Export + Versioning

In [ ]:
# ─── Version string ───────────────────────────────────────────────────────────
timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M")
version_id = f"v1_{timestamp}"

# ─── Save final model ─────────────────────────────────────────────────────────
model_filename = f"tomato_mobilenetv2_{version_id}.keras"
model_path     = MODELS_DIR / model_filename
model.save(model_path)
print(f"✅ Model saved: {model_path}")

# ─── model_config.yaml — consumed by the inference API ───────────────────────
model_config = {
    "version"             : version_id,
    "model_file"          : model_filename,
    "architecture"        : "MobileNetV2",
    "input_size"          : list(IMG_SIZE),
    "num_classes"         : NUM_CLASSES,
    "class_names"         : CLASS_NAMES,
    "preprocessing"       : "mobilenet_v2",    # tells API which preprocess_input to use
    "thresholds"          : {
        "confident"       : THRESHOLD_CONFIDENT,
        "uncertain"       : THRESHOLD_UNCERTAIN,
    },
    "trained_on"          : "PlantDoc (tomato subset)",
    "training_date"       : timestamp,
    "val_accuracy"        : round(float(best_val_acc), 4),
    "fine_tune_from_layer": FINE_TUNE_FROM,
}

import yaml  # kaggle has pyyaml pre-installed
config_path = MODELS_DIR / f"model_config_{version_id}.yaml"
with open(config_path, "w") as f:
    yaml.dump(model_config, f, default_flow_style=False, sort_keys=False)
print(f"✅ Config saved: {config_path}")

# ─── registry.json — append this version to the registry ─────────────────────
registry_path = MODELS_DIR / "registry.json"
registry = []
if registry_path.exists():
    with open(registry_path) as f:
        registry = json.load(f)

registry.append({
    **model_config,
    "per_class_metrics" : {
        cls: {
            "precision" : round(report_dict[cls]["precision"], 4),
            "recall"    : round(report_dict[cls]["recall"], 4),
            "f1"        : round(report_dict[cls]["f1-score"], 4),
            "support"   : int(report_dict[cls]["support"]),
        }
        for cls in CLASS_NAMES
    },
    "is_active" : True,   # set False on old entries before deploying a new version
})

with open(registry_path, "w") as f:
    json.dump(registry, f, indent=2)
print(f"✅ Registry updated: {registry_path}")

In [ ]:
# ─── Print summary of files to download ───────────────────────────────────────
print("\n" + "=" * 60)
print("📦 Files to download from Kaggle / Colab output:")
print("=" * 60)
for f in sorted(MODELS_DIR.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:<55} {size_mb:>7.2f} MB")
print()
for f in sorted(LOGS_DIR.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  logs/{f.name:<51} {size_mb:>7.2f} MB")
print()
print("Drop these into your local project:")
print("  → saved_models/<model_file>")
print("  → saved_models/registry.json")
print("  → training/configs/model_config_<version>.yaml")

## 11. Quick Inference Test (Sanity Check)
Run a few val images through the saved model to confirm it loads and predicts correctly.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenetv2_preprocess
from tensorflow.keras.preprocessing import image as keras_image

loaded_model = keras.models.load_model(model_path)
print(f"Model loaded: {model_path.name}")

def predict_single(img_path: Path, model, class_names: list, thresholds: dict):
    img = keras_image.load_img(img_path, target_size=IMG_SIZE)
    arr = keras_image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = mobilenetv2_preprocess(arr)

    probs      = model.predict(arr, verbose=0)[0]
    top_idx    = np.argsort(probs)[::-1]
    confidence = float(probs[top_idx[0]])
    pred_class = class_names[top_idx[0]]

    if confidence >= thresholds["confident"]:
        status = "CONFIDENT"
    elif confidence >= thresholds["uncertain"]:
        status = "UNCERTAIN"
    else:
        status = "FAILED"

    return {
        "predicted_class" : pred_class,
        "confidence"      : round(confidence, 4),
        "status"          : status,
        "top_k"           : [
            {"class": class_names[i], "confidence": round(float(probs[i]), 4)}
            for i in top_idx[:3]
        ],
    }

thresholds = {"confident": THRESHOLD_CONFIDENT, "uncertain": THRESHOLD_UNCERTAIN}

# Test on 5 random val images
sample_imgs = list(DATA_DIR.rglob("*.jpg"))[:5] + list(DATA_DIR.rglob("*.png"))[:5]
import random; random.shuffle(sample_imgs)

print(f"\n{'Image':<45} {'Pred':<30} {'Conf':>6} {'Status'}")
print("-" * 100)
for img_path in sample_imgs[:5]:
    result = predict_single(img_path, loaded_model, CLASS_NAMES, thresholds)
    print(f"{img_path.name:<45} {result['predicted_class']:<30} {result['confidence']:>6.3f}  {result['status']}")